In [10]:
import instructor
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List
import time

In [11]:

# Define the structure you want
class Step(BaseModel):
    """Describes a step in the AR Training Experience"""
    
    name: str = Field(
        description="A short, descriptive identifier for the step"
    )
    title: str = Field(
        description="A clear and concise task description, to be displayed"
    )
    description: str = Field(
        description="Detailed verbal instructions for the user, guiding them through the task to be performed"
    )

class ARLab(BaseModel):
    """A list of AR training steps"""
    steps: List[Step] = Field(
        description="Complete list of all the steps in the AR training experience"
    )


In [12]:

# Load the environment file with our api keys set
# create a .env file in the same folder, put your api key environment variables there
# Don't forget to put the .env in the gitignore so you don't publish your api keys!
load_dotenv()

# Connect to the LLM with Instructor
client = instructor.from_provider("openai/gpt-5-nano")


In [ ]:
prompt = """You are an AI agent tasked with generating step by step instructional scenarios for XR training experiences.
Each step is made of the following elements:
Name, Title, and Description.


Generate a training scenario by creating a sequence of steps to accomplish the objective:
Learn how to use the basic functions of a Carvey CNC machine to be able to use it for general operation.
The target audience is:
Students and staff at a university that have varying levels of familiarity with this technology.
The Activity that will be done to accomplish the objective is:
Use the Carvey to carve a stylized letter from wood.

Ensure each step is relevant, accurate, and practical for achieving the training objective within an XR environment.
"""

# start timer
start_time = time.time()

# Extract structured data
# Old method, without access to raw response
# lab = client.create(
# New method, which gives us access to token usage
lab, completion = client.create_with_completion(
    response_model=ARLab,
    messages=[
        {"role": "user", "content": prompt}
    ],
    max_retries=3
)

# end timer
end_time = time.time()

# total time taken
elapsed_time = end_time - start_time


In [14]:

# Price Calculation
gpt5n_input = 0.05  # Per 1M tokens
gpt5n_output = 0.40 # Per 1M tokens

input_tokens = completion.usage.prompt_tokens
output_tokens = completion.usage.completion_tokens
total_tokens = completion.usage.total_tokens

input_cost = input_tokens * gpt5n_input / 1000000
output_cost = output_tokens * gpt5n_output / 1000000
total_cost = input_cost + output_cost


In [16]:
# Print output
print(f"Elapsed Time : {elapsed_time}")
print()
print("Token Usage:")
print(f"input tokens  = {input_tokens}")
print(f"output tokens = {output_tokens}")
print(f"total tokens  = {total_tokens}")
print()
print(f"Total Cost : ${total_cost}")
print()
for i, step in enumerate(lab.steps):
    print(f"Step {i} : {step.name}\nTitle : {step.title}\nDescription : {step.description}\n")

Elapsed Time : 30.56895899772644

Token Usage:
input tokens  = 353
output tokens = 3111
total tokens  = 3464

Total Cost : $0.00126205

Step 0 : Intro_Safety
Title : Safety briefing and PPE
Description : Begin with a high-level safety overview of using the Carvey CNC. Identify potential hazards (moving parts, sharp bits, dust, loose clothing) and establish PPE requirements (safety glasses, dust mask if needed). Explain emergency stop location and how to halt a job quickly. Set expectations for XR training objectives and hands-on responsibilities.

Step 1 : Machinery_Overview
Title : Carvey components and workspace
Description : Tour the Carvey workstation: spindle nose, end mill, wasteboard, hold-downs, clamps, dust collection, limit switches, Carbide Motion controller and Carbide Create software. Explain the axes (X/Y/Z) and how they translate to the work surface.

Step 2 : Power_On_and_Estop
Title : Power on and emergency stop
Description : Power up the Carvey and the controller, ver